# പാഠം 10 - ഉല്പാദനത്തിലെ AI ഏജന്റുകൾ

ഈ പാഠത്തിൽ നിങ്ങൾ Microsoft Agent Framework (Python) ഉപയോഗിച്ച് AI ഏജന്റുകൾക്കുള്ള **ഉല്പാദന പാറ്റേണുകൾ** പഠിക്കും. ഞങ്ങൾ ഉൾപ്പെടുത്തുന്നു:

- **നിരീക്ഷണശേഷി** — ഏജന്റ് ഇടപെടലുകളിൽ സമയം രേഖപ്പെടുത്തലും ലോഗ്ഗിംഗും ചേർക്കൽ
- **മൂല്യനിർണയം** — പ്രതികരണ ഗുണനിലവാരം വിലയിരുത്താൻ ഒരു മൂല്യനിർണയ ഏജന്റ് ഉപയോഗിക്കൽ
- **ചെലവ് നിയന്ത്രണം** — ടോക്കൺ ഓപ്റ്റിമൈസേഷൻ, മോഡൽ തിരഞ്ഞെടുപ്പിനുള്ള തന്ത്രങ്ങൾ

ഈ സംഭവം ഒരു **ട്രാവൽ ഏജന്റ്** ആണ്, ഉപഭോക്താക്കളുടെ യാത്രാ പദ്ധതി സഹായിക്കുന്നതും നിരീക്ഷണവും മൂല്യനിർണയവും മുകളിലായി ഉൾപ്പെടുത്തിയതാണ്.


## ക്രമീകരണം


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ഉത്പാദന പരിഗണനകൾ

പ്രോട്ടോടൈപ്പുകളിൽ നിന്ന് AI ഏജന്റുകളെ ഉത്പാദനത്തിലേക്ക് മാറ്റുന്നത് മൂന്ന് പൈലറുകളിൽ ശ്രദ്ധ നൽകേണ്ടതാണ്:

1. **അവലോകനം** — ഏജന്റ് എന്ത് ചെയ്യുന്നു, അതിന് എത്ര സമയം എടുക്കുന്നു, ഏത് ഉപകരണങ്ങൾ വിളിക്കുന്നു എന്നിവ കാണാൻ ദൃശ്യത വേണം. ട്രേസിംഗും ലോഗ്ഗിംഗും ഇല്ലാതെ, ഉത്പാദന പ്രശ്നങ്ങൾ ഡീബગીং ചെയ്യുന്നത് പ്രായോഗികമായി അസാധ്യമാണ്.

2. **മൂല്യനിര്‍ണ്ണയം** — സ്വയംക്രമിത ഗുണനിലവാര പരിശോധനകൾ ഏജന്റിന്റെ മറുപടികൾ സമയത്തേക്ക് കൃത്യവും, സമ്പൂർണ്ണവും, ഉപകാരപ്രദവുമാകുന്നതായി ഉറപ്പാക്കുന്നു. ഒരു മൂല്യനിർണയ ഏജന്റ് നിർവചിച്ച മാനദണ്ഡങ്ങൾ അനുസരിച്ചു മറുപടികൾ സ്കോർ ചെയ്യാവുന്നതാണ്.

3. **ചെലവ് നിയന്ത്രണം** — ടോക്കെൻ ഉപയോഗം നേരിട്ട് ചെലവിനെ ബാധിക്കുന്നു. പ്രാമ്പ്റ്റ് ഒപ്റ്റിമൈസേഷൻ, മോഡൽ തിരഞ്ഞെടുപ്പ്, കാഷിംഗ് തുടങ്ങിയതുപോലെ തന്ത്രങ്ങൾ ഗുണനിലവാരം നഷ്ടപ്പെടാതെ ചെലവ് നിയന്ത്രിക്കാൻ സഹായിക്കുന്നു.


## ഒരു നിരീക്ഷണ ഏജന്റിനെ നിർമ്മിക്കൽ

നാം യാത്രാ ഉപകരണങ്ങൾ నిర్వചിച്ച് ഏജന്റ് കോൾ സമയമർദ്ദനത്തോടുകൂടെ ചുറ്റിപ്പറ്റുന്നു, അതിലൂടെ വില്ലമ്പതിയെ നാം നിരീക്ഷിക്കാൻ കഴിയും. ഉത്പാദനത്തിൽ നിങ്ങൾ OpenTelemetry അല്ലെങ്കിൽ സമാനമായ ഒരു ട്രേസിംഗ് ബാക്ക്‌എൻഡിനെ ഇന്റഗ്രേറ്റ് ചെയ്യും.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## വിലയിരുത്തൽ കേസുകൾ

സാധാരണ ഉത്പാദന മാതൃകയിൽ, രണ്ടാമത്തെ ഏജന്റ് **വിലയിരുത്തിയ হিসাবে** ഉപയോഗിക്കുന്നു. വിലയിരുത്തിയ ആദ്യത്തെ ഏജന്റ് നൽകിയ പ്രതികരണം പൂർണത, കൃത്യത, സഹായകത പോലുള്ള മുൻകൂട്ടി നിർവ്വചിച്ച മാനദണ്ഡങ്ങൾ അടിസ്ഥാനമാക്കി സ്കോർ ചെയ്യുന്നു.

ഇതുവഴി സാധ്യമാണ്:
- ഉപയോക്താക്കൾക്ക് പരിച്ഛേദം കിട്ടുന്നതിന് മുമ്പുള്ള സ്വയം പ്രവർത്തന ഗുണനിലവാര നിരീക്ഷണം
- പ്രോംപ്റ്റുകൾ അല്ലെങ്കിൽ മോഡലുകൾ മാറുമ്പോൾ പുനരാവർത്തന കണ്ടെത്തൽ
- ഏജന്റിന്റെ പ്രകടനം സമയാന്തരമായി തുടർച്ചയായ നിരീക്ഷണം


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## ചെലവ് കൈകാര്യം ചെയ്യൽ തന്ത്രങ്ങൾ

ഉൽപാദന AI ഏജൻറുകൾക്ക് ചെലവുകൾ നിയന്ത്രിക്കുക സുപ്രധാനമാണ്. ഇത aquí പ്രധാന തന്ത്രങ്ങൾ:

| തന്ത്രം | വിശദീകരണം |
|---|---|
| **പ്രോംപ്റ്റ് ഒപ്റ്റിമൈസേഷൻ** | സിസ്റ്റം നിർദ്ദേശങ്ങൾ ചുരുക്കി വെക്കുക. ഇൻപുട്ട് ടോക്കണുകൾ കുറയ്ക്കുന്നതിന് ആവൃത്തി ഉള്ള കോൺടെക്സ്റ്റ് നീക്കം ചെയ്യുക. |
    "| **മോഡൽ തിരഞ്ഞെടുപ്പ്** | ക്ലാസിഫിക്കേഷൻ അല്ലെങ്കിൽ എക്‌സ്‌ട്രാക്ഷൻ പോലുള്ള ലളിതമായ ജോലികൾക്ക് ചെറിയ, കുറഞ്ഞ ചെലവുള്ള മോഡലുകൾ (ഉദാ: GPT-5-മിനി) ഉപയോഗിക്കുക, സങ്കീർണ്ണമായ തർക്കങ്ങൾക്ക് വലിയ മോഡലുകൾ محفوظമാക്കുക. |\n",
| **കാഷിങ്** | ഉപകരണം ഫലങ്ങളും പതിവ് ക്വ്റെറിയുകളും കാഷ് ചെയ്യുക, ആവർത്തന API കോൾകൾ ഒഴിവാക്കാൻ. |
| **ടോക്കൺ ബജറ്റുകൾ** | പ്രതീക്ഷിക്കാത്ത ദൈർഘ്യമുള്ള മറുപടികൾ തടയാൻ `max_tokens` പരിധി നിശ്ചയിക്കുക. |
| **ബാച്ചിംഗ്** | സാധ്യമായിടത്ത് നിരവധി ഉപയോക്തൃ ക്വ്റെറികളെ ഒറ്റ API കോളിൽ കൂട്ടിക്കുക. |

പ്രായോഗികമായി, ഒരു നിര പ്രസ്താവന നല്ല രീതിയിൽ പ്രവർത്തിക്കുന്നു: ലളിതമായ അഭ്യർത്ഥനകൾ വേഗമായ, ചിലവു കുറഞ്ഞ മോഡലിലേക്ക് വഴിമാറിക്കുകയും, സങ്കീർണ്ണമായ ക്വ്റെറികൾ മാത്രമേ കൂടുതൽ കഴിവുള്ള മോഡലിലേക്ക് ഉയർത്തപ്പെടുകയുള്ളൂ.


## സംക്ഷേപം

ഈ പാഠത്തിൽ നിങ്ങൾ പഠിച്ചത്:

1. **സമയം ശ്രദ്ധിച്ച് ലോഗ് ചെയ്യലും ഉൾപ്പെടെ ഏജന്റുകളുടെ ഇടപെടലുകളിൽ അവലോകനക്ഷമത추ം ചേർക്കുക**, ട്രേസിംഗിനും മോണിറ്ററിംഗിനും അടിസ്ഥാനമിടുന്നു.
2. **ഏജന്റ് പ്രതികരണങ്ങൾ സ്വയം വിലയിരുത്തുക**, പൂർത്തിയാകൽ, സത്യസന്ധത, സഹായകരത എന്നിവയ്ക്ക് സ്‌കോർ നൽകുന്ന ഒരു മൂല്യനിർണയ ഏജന്റ് ഉപയോഗിച്ച്.
3. **ചെലവുകൾ നിയന്ത്രിക്കുക** പ്രോംപ്റ്റ് അനുയോജിതമാക്കൽ, മോഡൽ തിരഞ്ഞെടുപ്പ്, കാഷിങ്ങ്, ടോക്കൺ ബജറ്റുകൾ എന്നിവയിലൂടെ.

ഈ ഉല്‍പ്പാദന മാതൃകകൾ നിങ്ങളുടെ AI ഏജൻറുകൾ വിശ്വസനീയവും, അളക്കാനും, ചെലവിൽ കാര്യക്ഷമവുമാക്കുന്നതിന് സഹായിക്കുന്നു.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
